# 🫁 Chest X-Ray Diagnostic Assistant
## Notebook 02 — Grad-CAM : Explicabilité du modèle

> **Prérequis** : Le Notebook 01 doit avoir été exécuté et `notebooks/best_resnet18.pt` doit exister.

---
```
PLAN DU NOTEBOOK
──────────────────────────────────────────────────────────
PARTIE 1 │ Pourquoi l'explicabilité est indispensable en médecine
PARTIE 2 │ Principe mathématique de Grad-CAM
PARTIE 3 │ Chargement du modèle et implémentation
PARTIE 4 │ Analyse de 3 cas : correct PNEUMONIA, correct NORMAL, erreur
──────────────────────────────────────────────────────────
```

In [ ]:
# ── Vérification des dépendances ─────────────────────────────────────────────
# Si pytorch-grad-cam n'est pas installé :
# pip install grad-cam

import subprocess, sys
try:
    import pytorch_grad_cam
    print('✅ pytorch-grad-cam déjà installé')
except ImportError:
    print('Installation de grad-cam...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'grad-cam', '-q'])
    print('✅ grad-cam installé')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import cv2
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Chemins
DATA_ROOT  = Path(r'D:\brave\PROJECT LINKEDIN\archive\chest_xray')
MODEL_PATH = Path(r'D:\brave\PROJECT LINKEDIN\notebooks\best_resnet18.pt')

assert DATA_ROOT.exists(),  f'❌ Dataset introuvable : {DATA_ROOT}'
assert MODEL_PATH.exists(), f'❌ Modèle introuvable : {MODEL_PATH} (exécute le Notebook 01 d\'abord)'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASS_NAMES   = ['NORMAL', 'PNEUMONIA']

print(f'✅ Device : {DEVICE}')
print(f'✅ Dataset : {DATA_ROOT}')
print(f'✅ Modèle  : {MODEL_PATH}')

---
# 🩺 PARTIE 1 — Pourquoi l'explicabilité est indispensable en médecine

## Le problème de la "boîte noire"

Notre ResNet-18 du Notebook 01 atteint des performances impressionnantes (>95% d'accuracy).  
Mais en médecine, **savoir qu'un modèle est précis ne suffit pas**.

### Imagine ce scénario :

```
Modèle : "PNEUMONIA — 97% de confiance"
Médecin : "Mais... il regarde QUOI pour prendre cette décision ?"
```

Sans explicabilité, **impossible de savoir** si le modèle :
- ✅ Détecte les vraies opacités/consolidations pulmonaires (correct)
- ❌ Réagit à un cathéter, une étiquette de radiologie, ou un artefact (biais)

### Exemple documenté de biais IA en médecine :
> Un modèle de détection de mélanome atteignait 90%+ d'accuracy...  
> mais avait appris que les images contenant une règle de mesure dermato étaient plus souvent des mélanomes  
> → Il "diagnostiquait" la règle, pas la tumeur. *(Winkler et al., 2019)*

### La solution : Grad-CAM
→ Visualiser exactement **quelles zones de l'image** ont influencé la décision du modèle.

---
# 🧮 PARTIE 2 — Principe mathématique de Grad-CAM

## Intuition

Rappelle-toi : ResNet-18 est une succession de couches convolutives.  
La **dernière couche convolutive** (`layer4`) produit des feature maps — chacune représente un détecteur de motif spécialisé.

**Idea** : Quelles feature maps ont le plus influencé la prédiction finale ?  
→ On utilise les **gradients** pour le savoir.

## Formule de Grad-CAM

**Étape 1** : Calculer le gradient de la classe cible $y^c$ par rapport à chaque feature map $A^k$ :
$$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{ij}^k}$$

→ $\alpha_k^c$ = **importance** de la feature map $k$ pour prédire la classe $c$

**Étape 2** : Combiner les feature maps pondérées par leur importance :
$$L^c_{\text{Grad-CAM}} = \text{ReLU}\left( \sum_k \alpha_k^c \cdot A^k \right)$$

→ ReLU car on ne garde que les influences **positives** (zones qui "poussent" vers la classe $c$)

**Résultat** : une carte de chaleur (heatmap) de la taille de la feature map → interpolée à la taille originale.

```
Image 224×224 → ResNet-18 → feature map 7×7 (layer4) → heatmap 7×7 → resize 224×224
```

In [ ]:
# ── 2.1 Illustration schématique du pipeline Grad-CAM ─────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('#f0f0f0')
ax.set_xlim(0, 18); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('Pipeline Grad-CAM : de l\'image à la heatmap d\'attention', fontsize=13, fontweight='bold')

steps = [
    (1.5, '#4CAF50', 'Image\n224×224\nX-Ray'),
    (5,   '#2196F3', 'ResNet-18\nBackbone\n(conv layers)'),
    (8.5, '#9C27B0', 'Feature Maps\nlayer4\n7×7×512'),
    (12,  '#F44336', 'Gradients\n∂y_class /\n∂feature_maps'),
    (15.5,'#FF9800', 'Heatmap\n7×7 → 224×224\n(Grad-CAM)'),
]
for x, color, label in steps:
    ax.add_patch(mpatches.FancyBboxPatch((x-1.2, 1.5), 2.4, 3,
                  boxstyle='round,pad=0.1', facecolor=color, alpha=0.8, edgecolor='white', lw=2))
    ax.text(x, 3, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

arrow_xs = [(2.8,3.8),(6.3,7.3),(9.8,10.8),(13.3,14.3)]
for x1, x2 in arrow_xs:
    ax.annotate('', xy=(x2, 3), xytext=(x1, 3),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Annotations formule
ax.text(8.5, 0.8, r'$\alpha_k^c = \frac{1}{Z}\sum_{ij}\frac{\partial y^c}{\partial A_{ij}^k}$',
        ha='center', fontsize=11, color='#9C27B0')
ax.text(12, 0.8, r'$L^c = \mathrm{ReLU}(\sum_k \alpha_k^c \cdot A^k)$',
        ha='center', fontsize=11, color='#F44336')

plt.tight_layout()
plt.show()

---
# 🔧 PARTIE 3 — Chargement du modèle et implémentation Grad-CAM

In [ ]:
# ── 3.1 Chargement du modèle sauvegardé ──────────────────────────────────────
model = resnet18(weights=None)
model.fc = nn.Linear(512, 2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

print('Architecture de la couche cible pour Grad-CAM :')
print(f'  → model.layer4[-1]  (dernière couche convolutive de ResNet-18)')
print(f'  → Produit des feature maps 7×7×512')
print(f'\n✅ Modèle chargé avec succès — {sum(p.numel() for p in model.parameters()):,} paramètres')

In [ ]:
# ── 3.2 Dataset et utilitaires ────────────────────────────────────────────────
tfm_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_ds = datasets.ImageFolder(str(DATA_ROOT / 'test'), tfm_eval)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

# ── Utilitaires de visualisation ─────────────────────────────────────────────
def denormalize(tensor):
    """Dé-normalise un tensor image pour l'affichage (retourne numpy [0,1])."""
    m = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    s = torch.tensor(IMAGENET_STD).view(3,1,1)
    img = (tensor * s + m).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

def get_prediction(model, tensor, device):
    """Retourne (class_idx, confidence, class_name) pour un tensor [1,3,H,W]."""
    with torch.no_grad():
        out = model(tensor.unsqueeze(0).to(device))
        probs = torch.softmax(out, dim=1).squeeze().cpu()
    pred_idx = probs.argmax().item()
    return pred_idx, probs[pred_idx].item(), CLASS_NAMES[pred_idx]

print(f'Test dataset : {len(test_ds)} images')
print(f'Classes : {test_ds.classes}')

In [ ]:
# ── 3.3 Sélection des 3 cas d'analyse ────────────────────────────────────────
# On parcourt le test set pour trouver :
#   • Un vrai positif (PNEUMONIA prédit correctement)
#   • Un vrai négatif (NORMAL prédit correctement)
#   • Une erreur (prédit incorrectement)

cases = {'correct_pneumonia': None, 'correct_normal': None, 'incorrect': None}

for idx in range(len(test_ds)):
    tensor, true_label = test_ds[idx]
    pred_idx, conf, pred_name = get_prediction(model, tensor, DEVICE)
    true_name = CLASS_NAMES[true_label]
    correct = (pred_idx == true_label)

    if correct and true_label == 1 and cases['correct_pneumonia'] is None and conf > 0.92:
        cases['correct_pneumonia'] = (idx, tensor, true_label, pred_idx, conf)
    elif correct and true_label == 0 and cases['correct_normal'] is None and conf > 0.92:
        cases['correct_normal'] = (idx, tensor, true_label, pred_idx, conf)
    elif not correct and cases['incorrect'] is None:
        cases['incorrect'] = (idx, tensor, true_label, pred_idx, conf)

    if all(v is not None for v in cases.values()):
        break

print('✅ 3 cas sélectionnés :')
labels_desc = {
    'correct_pneumonia': '🔴 PNEUMONIA prédit correctement',
    'correct_normal': '🟢 NORMAL prédit correctement',
    'incorrect': '⚠️  Prédiction INCORRECTE',
}
for key, (idx, tensor, true_lbl, pred_idx, conf) in cases.items():
    print(f'  {labels_desc[key]}')
    print(f'     Vrai label : {CLASS_NAMES[true_lbl]} | Prédit : {CLASS_NAMES[pred_idx]} ({conf:.1%})')

In [ ]:
# ── 3.4 Fonction Grad-CAM ─────────────────────────────────────────────────────
def compute_gradcam(model, tensor, target_class, device):
    """
    Calcule la heatmap Grad-CAM pour une image et une classe cible.
    
    Args:
        model       : ResNet-18 chargé
        tensor      : image normalisée [3, H, W]
        target_class: index de la classe cible (0=NORMAL, 1=PNEUMONIA)
        device      : cpu ou cuda
    Returns:
        heatmap    : np.array [H, W] en float [0,1] (heatmap brute)
        img_rgb    : np.array [H, W, 3] float [0,1] (image dénormalisée)
        cam_image  : np.array [H, W, 3] float [0,1] (heatmap superposée)
    """
    # Cible : dernière couche convolutive de ResNet-18 (layer4[-1])
    target_layers = [model.layer4[-1]]
    cam = GradCAM(model=model, target_layers=target_layers)

    input_tensor = tensor.unsqueeze(0).to(device)  # [1, 3, 224, 224]
    targets = [ClassifierOutputTarget(target_class)]

    # Calcul Grad-CAM
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    heatmap = grayscale_cam[0]  # [224, 224], valeurs [0,1]

    # Image RGB dénormalisée pour la superposition
    img_rgb = denormalize(tensor)  # [224, 224, 3], float [0,1]

    # Superposition heatmap + image
    cam_image = show_cam_on_image(img_rgb, heatmap, use_rgb=True, image_weight=0.5)
    cam_image = cam_image.astype(np.float32) / 255.0

    return heatmap, img_rgb, cam_image

def get_heatmap_location(heatmap, threshold=0.6):
    """
    Analyse la position de la zone d'attention dominante dans la heatmap.
    Retourne une description textuelle de la zone (base, apex, centre, diffuse).
    """
    h, w = heatmap.shape
    mask = heatmap >= threshold

    if mask.sum() == 0:
        return 'diffuse (aucune zone d\'attention dominante)', 0.5, 0.5

    rows, cols = np.where(mask)
    cy = rows.mean() / h  # Centre vertical normalisé (0=haut, 1=bas)
    cx = cols.mean() / w  # Centre horizontal normalisé (0=gauche, 1=droite)

    if cy < 0.33:
        zone_v = 'apex (zone supérieure)'
    elif cy > 0.66:
        zone_v = 'base (zone inférieure)'
    else:
        zone_v = 'zone médiane (hilaire)'

    if cx < 0.4:
        zone_h = 'gauche'
    elif cx > 0.6:
        zone_h = 'droite'
    else:
        zone_h = 'bilatérale/centrale'

    return f'{zone_v}, côté {zone_h}', cy, cx

print('✅ Fonctions Grad-CAM prêtes')

---
# 🔬 PARTIE 4 — Analyse des 3 cas

Pour chaque cas : **Image originale | Heatmap seule | Superposition**

In [ ]:
# ── 4.1 CAS 1 : Prédiction correcte PNEUMONIA ────────────────────────────────
idx, tensor, true_lbl, pred_idx, conf = cases['correct_pneumonia']

heatmap, img_rgb, cam_image = compute_gradcam(model, tensor, pred_idx, DEVICE)
zone_desc, cy, cx = get_heatmap_location(heatmap)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle(
    f'CAS 1 — ✅ Prédiction CORRECTE : PNEUMONIA ({conf:.1%})\n'
    f'Zone d\'attention : {zone_desc}',
    fontsize=13, fontweight='bold', color='#c62828'
)

axes[0].imshow(img_rgb); axes[0].set_title('Image originale\n(radio thoracique)', fontsize=11); axes[0].axis('off')
im_h = axes[1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
axes[1].set_title('Heatmap Grad-CAM\n(rouge = forte attention)', fontsize=11); axes[1].axis('off')
plt.colorbar(im_h, ax=axes[1], shrink=0.8, label='Intensité d\'attention')
axes[2].imshow(cam_image); axes[2].set_title('Superposition\n(zones rouges = décision du modèle)', fontsize=11); axes[2].axis('off')

plt.tight_layout()
plt.show()

print('ANALYSE CAS 1 :')
print(f'  Vraie classe  : {CLASS_NAMES[true_lbl]}')
print(f'  Prédiction    : {CLASS_NAMES[pred_idx]} ({conf:.1%})')
print(f'  Zone attention: {zone_desc}')
print()
print('  → La heatmap devrait se concentrer sur les zones d\'opacité/consolidation')
print('  → Si elle couvre les champs pulmonaires inférieurs/moyens : modèle cohérent!')
print('  → Si elle est sur les bords de l\'image ou des artefacts : biais possible')

In [ ]:
# ── 4.2 CAS 2 : Prédiction correcte NORMAL ────────────────────────────────────
idx, tensor, true_lbl, pred_idx, conf = cases['correct_normal']

heatmap, img_rgb, cam_image = compute_gradcam(model, tensor, pred_idx, DEVICE)
zone_desc, cy, cx = get_heatmap_location(heatmap)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle(
    f'CAS 2 — ✅ Prédiction CORRECTE : NORMAL ({conf:.1%})\n'
    f'Zone d\'attention : {zone_desc}',
    fontsize=13, fontweight='bold', color='#1b5e20'
)

axes[0].imshow(img_rgb); axes[0].set_title('Image originale\n(poumon sain)', fontsize=11); axes[0].axis('off')
im_h = axes[1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
axes[1].set_title('Heatmap Grad-CAM\n(attention pour classe NORMAL)', fontsize=11); axes[1].axis('off')
plt.colorbar(im_h, ax=axes[1], shrink=0.8)
axes[2].imshow(cam_image); axes[2].set_title('Superposition', fontsize=11); axes[2].axis('off')

plt.tight_layout()
plt.show()

print('ANALYSE CAS 2 :')
print(f'  Vraie classe  : {CLASS_NAMES[true_lbl]}')
print(f'  Prédiction    : {CLASS_NAMES[pred_idx]} ({conf:.1%})')
print(f'  Zone attention: {zone_desc}')
print()
print('  → Pour un poumon NORMAL, la heatmap est souvent plus diffuse')
print('  → Le modèle peut regarder les grandes structures (côtes, diaphragme)')
print('    pour CONFIRMER l\'ABSENCE d\'opacités')
print('  → Une heatmap uniforme = le modèle inspecte l\'ensemble des champs pulmonaires')

In [ ]:
# ── 4.3 CAS 3 : Prédiction INCORRECTE (analyse du biais) ─────────────────────
idx, tensor, true_lbl, pred_idx, conf = cases['incorrect']

# Grad-CAM pour la classe PRÉDITE (incorrecte) et la VRAIE classe
heatmap_pred, img_rgb, cam_pred = compute_gradcam(model, tensor, pred_idx, DEVICE)
heatmap_true, _,       cam_true = compute_gradcam(model, tensor, true_lbl, DEVICE)

zone_pred, cy_p, cx_p = get_heatmap_location(heatmap_pred)
zone_true, cy_t, cx_t = get_heatmap_location(heatmap_true)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    f'CAS 3 — ❌ Prédiction INCORRECTE\n'
    f'Vraie classe : {CLASS_NAMES[true_lbl]} | Prédit : {CLASS_NAMES[pred_idx]} ({conf:.1%})',
    fontsize=13, fontweight='bold', color='#e65100'
)

# Ligne 1 : Grad-CAM pour la classe PRÉDITE
axes[0,0].imshow(img_rgb); axes[0,0].set_title('Image originale', fontsize=11); axes[0,0].axis('off')
im1 = axes[0,1].imshow(heatmap_pred, cmap='jet', vmin=0, vmax=1)
axes[0,1].set_title(f'Grad-CAM → classe PRÉDITE ({CLASS_NAMES[pred_idx]})\n{zone_pred}', fontsize=10, color='#c62828')
axes[0,1].axis('off'); plt.colorbar(im1, ax=axes[0,1], shrink=0.8)
axes[0,2].imshow(cam_pred); axes[0,2].set_title(f'Superposition : là où le modèle\n"a vu" du {CLASS_NAMES[pred_idx]}', fontsize=10); axes[0,2].axis('off')

# Ligne 2 : Grad-CAM pour la VRAIE classe
axes[1,0].imshow(img_rgb); axes[1,0].set_title('Même image', fontsize=11); axes[1,0].axis('off')
im2 = axes[1,1].imshow(heatmap_true, cmap='jet', vmin=0, vmax=1)
axes[1,1].set_title(f'Grad-CAM → vraie classe ({CLASS_NAMES[true_lbl]})\n{zone_true}', fontsize=10, color='#1b5e20')
axes[1,1].axis('off'); plt.colorbar(im2, ax=axes[1,1], shrink=0.8)
axes[1,2].imshow(cam_true); axes[1,2].set_title(f'Superposition : ce qui aurait dû\nle faire classer {CLASS_NAMES[true_lbl]}', fontsize=10); axes[1,2].axis('off')

plt.tight_layout()
plt.show()

print('ANALYSE CAS 3 — Autopsie de l\'erreur :')
print(f'  Vraie classe  : {CLASS_NAMES[true_lbl]}')
print(f'  Prédiction    : {CLASS_NAMES[pred_idx]} ({conf:.1%})')
print()
print(f'  Grad-CAM classe prédite ({CLASS_NAMES[pred_idx]}) → {zone_pred}')
print(f'  Grad-CAM vraie classe   ({CLASS_NAMES[true_lbl]}) → {zone_true}')
print()
print('  INTERPRÉTATION :')
if true_lbl == 0:  # Vraie classe NORMAL, prédit PNEUMONIA
    print('  → Faux positif : le modèle a détecté une zone "suspecte" dans un poumon sain')
    print('  → Causes possibles : artefact de compression JPEG, ombre anatomique, positionnement patient')
    print('  → La Grad-CAM ligne 1 montre où le modèle a "halluciné" une opacité')
else:  # Vraie classe PNEUMONIA, prédit NORMAL
    print('  → Faux négatif : le modèle a MANQUÉ une pneumonie (cas le plus dangereux !)')
    print('  → Causes possibles : pneumonie débutante ou discrète, infiltrats peu marqués')
    print('  → La Grad-CAM ligne 2 montre les zones d\'opacité que le modèle n\'a pas su exploiter')

In [ ]:
# ── 4.4 Figure récapitulative des 3 cas côte à côte ──────────────────────────
all_cases = [
    ('CAS 1\n✅ PNEUMONIA correct', cases['correct_pneumonia'], '#c62828'),
    ('CAS 2\n✅ NORMAL correct',    cases['correct_normal'],    '#1b5e20'),
    ('CAS 3\n❌ Erreur',            cases['incorrect'],          '#e65100'),
]

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
fig.suptitle('Récapitulatif Grad-CAM — 3 cas analysés', fontsize=14, fontweight='bold')

col_titles = ['Image originale', 'Heatmap Grad-CAM', 'Superposition']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold', pad=10)

for row, (case_title, case_data, color) in enumerate(all_cases):
    _, tensor, true_lbl, pred_idx, conf = case_data
    hm, img_rgb, cam_img = compute_gradcam(model, tensor, pred_idx, DEVICE)
    zone_desc, _, _ = get_heatmap_location(hm)

    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_ylabel(case_title, fontsize=10, color=color, fontweight='bold', rotation=0, labelpad=80, va='center')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(hm, cmap='jet', vmin=0, vmax=1)
    axes[row, 1].set_xlabel(f'Attention → {zone_desc}', fontsize=8, color=color)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(cam_img)
    correct_str = '✅' if pred_idx == true_lbl else '❌'
    axes[row, 2].set_xlabel(f'{correct_str} {CLASS_NAMES[true_lbl]} → {CLASS_NAMES[pred_idx]} ({conf:.0%})', fontsize=9, color=color)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(str(MODEL_PATH.parent / 'gradcam_3cases.png'), dpi=120, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée : notebooks/gradcam_3cases.png')

In [ ]:
# ── 4.5 Sauvegarder les cas pour le Notebook 03 ──────────────────────────────
import pickle

save_data = {
    key: {
        'tensor':    val[1],
        'true_label':val[2],
        'pred_idx':  val[3],
        'confidence':val[4],
    }
    for key, val in cases.items()
}

# Calculer et sauvegarder aussi les heatmaps et zones
for key, val in cases.items():
    _, tensor, true_lbl, pred_idx, conf = val
    hm, img_rgb, cam_img = compute_gradcam(model, tensor, pred_idx, DEVICE)
    zone_desc, cy, cx = get_heatmap_location(hm)
    save_data[key]['heatmap']    = hm
    save_data[key]['img_rgb']    = img_rgb
    save_data[key]['cam_image']  = cam_img
    save_data[key]['zone_desc']  = zone_desc
    save_data[key]['zone_cy']    = cy
    save_data[key]['zone_cx']    = cx

pkl_path = MODEL_PATH.parent / 'gradcam_cases.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(save_data, f)

print(f'✅ Données Grad-CAM sauvegardées : {pkl_path.name}')
print('   → Sera chargé par le Notebook 03 (RAG)')
print()
print('RÉSUMÉ DES 3 CAS :')
print('─' * 60)
for key, d in save_data.items():
    correct = '✅' if d['pred_idx'] == d['true_label'] else '❌'
    print(f'  {correct} {key}')
    print(f'     Vrai:{CLASS_NAMES[d["true_label"]]} | Prédit:{CLASS_NAMES[d["pred_idx"]]} ({d["confidence"]:.1%}) | Zone:{d["zone_desc"]}')